# 01 — California Housing EDA

Exploratory Data Analysis (EDA) for the California Housing dataset.

This notebook:
- loads the raw dataset snapshot
- inspects schema and missingness
- visualizes target distribution
- checks feature correlations
- plots relationships with the target

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'california_housing.csv'

print('Project root:', PROJECT_ROOT)
print('Raw data path:', RAW_PATH)

In [ ]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f'{RAW_PATH} not found. Run `python src/data.py` from project root first.'
    )

df = pd.read_csv(RAW_PATH)
df.head()

In [ ]:
print('Shape:', df.shape)
display(df.describe().T)

missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame(name='missing_count'))

In [ ]:
target = 'MedHouseVal'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[target], bins=40, kde=True, ax=axes[0], color='royalblue')
axes[0].set_title('Target Distribution: MedHouseVal')

sns.boxplot(x=df[target], ax=axes[1], color='lightseagreen')
axes[1].set_title('Target Boxplot')

plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap')
plt.show()

corr_with_target = corr[target].sort_values(ascending=False)
corr_with_target

In [ ]:
top_features = [f for f in corr_with_target.index if f != target][:4]
top_features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, feat in enumerate(top_features):
    sns.scatterplot(data=df.sample(min(4000, len(df)), random_state=42),
                    x=feat, y=target, alpha=0.4, s=20, ax=axes[i])
    axes[i].set_title(f'{feat} vs {target}')

plt.tight_layout()
plt.show()

## Notes

Use these findings to guide feature engineering and model choices.

Potential next steps:
- log transforms for skewed variables
- quantile clipping for outliers
- geo-feature interactions (Latitude × Longitude neighborhoods)
- cross-validation for robust model comparison